In [45]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.ica_pipeline import *

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [31]:
DATA_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/crystal"

NOISE_COV = None                      # Noise covariance used for pre-whitening. If None (default), channels are scaled to unit variance (“z-standardized”) as a group by channel type prior to the whitening by PCA.
RANDOM_STATE = 42                     # Fix this throughout the script -- always!!

METHOD = "infomax"                    # MNE accepts 'fastica' | 'infomax' | 'picard'; using "picard" to match EEGLAB default -> need to pip install python-picard
FIT_PARAMS={"extended": True,         # EEGLAB default is infomax extended
            "weights": None,          # The initialized unmixing matrix. Defaults to None, which means the identity matrix is used.
            "l_rate": None,           # This quantity indicates the relative size of the change in weights. Defaults to 0.01 / log(n_features ** 2).
            "block": None,            # The block size of randomly chosen data segments. Defaults to floor(sqrt(n_times / 3.)).           
            "w_change": 1e-12,        # The change at which to stop iteration. Defaults to 1e-12.
            "anneal_deg": 60.0,       # The angle (in degrees) at which the learning rate will be reduced. Defaults to 60.0.
            "anneal_step": 0.9,       # The factor by which the learning rate will be reduced. Defaults to 0.9.
            "n_subgauss": 1,          # The number of subgaussian components. Only considered for extended Infomax. Defaults to 1.
            "kurt_size": 6000,        # The window size for kurtosis estimation. Only considered for extended Infomax. Defaults to 6000.
            "blowup": 10000}          # The maximum difference allowed between two successive estimations of the unmixing matrix. Defaults to 10000.

# for picard
#FIT_PARAMS={"tol": 1e-7,
            #"ortho": False, # If True, uses Picard-O. Otherwise, uses the standard Picard.
            #"fastica_it": None} # If an int, perform fastica_it iterations of FastICA before running Picard. It might help starting from a better point.


MAX_ITER=1000 
#allow_ref_meg -> irrelevant

CORR_THRESHOLD = 0.85
VERBOSE=True 

In [ ]:
raw_clean_list, eog_indices_list = [], []
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_reref_filt.set"
    try:
        raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue

    raw_reref, raw_for_ica, rank = preprocess_for_ica(raw,
                                                  standard_montage="standard_1020",
                                                  eog_channels=["HEO", "VEO"],
                                                  ref_channels=["M1", "M2"],
                                                  time_threshold=3.0,
                                                  after_event_code_buffer=1.5,
                                                  before_event_code_buffer=0.5,
                                                  rank_tol=1e-4,
                                                  verbose=False)
    raw_clean, eog_indices, _ = perform_ica(raw_to_clean=raw_reref,
                                            raw_for_ica=raw_for_ica,
                                            n_components=rank,
                                            noise_cov=NOISE_COV,
                                            method=METHOD,
                                            fit_params=FIT_PARAMS,
                                            max_iter=MAX_ITER,
                                            manual_inspection=False,
                                            corr_threshold=CORR_THRESHOLD,
                                            eog_like_channels=["VEO"], 
                                            verbose=False)
    raw_clean_list.append(raw_clean)
    eog_indices_list.append(eog_indices)


Input raw file has missing channel positions; setting channel montage.
Found 5 boundary event(s) in the raw dataset:
[OrderedDict([('onset', np.float64(341.0795)), ('duration', np.float64(0.0)), ('description', np.str_('boundary')), ('orig_time', None), ('extras', {})]), OrderedDict([('onset', np.float64(659.5995)), ('duration', np.float64(0.0)), ('description', np.str_('boundary')), ('orig_time', None), ('extras', {})]), OrderedDict([('onset', np.float64(977.0395)), ('duration', np.float64(0.0)), ('description', np.str_('boundary')), ('orig_time', None), ('extras', {})]), OrderedDict([('onset', np.float64(1291.6395)), ('duration', np.float64(0.0)), ('description', np.str_('boundary')), ('orig_time', None), ('extras', {})]), OrderedDict([('onset', np.float64(1605.2795)), ('duration', np.float64(0.0)), ('description', np.str_('boundary')), ('orig_time', None), ('extras', {})])]
Applying a custom ('EEG',) reference.
Creating RawArray with float64 data, n_channels=35, n_times=1901840
    

In [44]:
# sanity checking channel positions are set and channel names are upper case
m = raw_clean.get_montage()
pos = m.get_positions()["ch_pos"]
print(pos["FP1"])

[-0.0309026   0.11458518  0.02786657]
